# MIL-CREDA frente a CREDA — fase uno: la escalera

Diez métodos sobre seis transferencias. Cada peldaño se diferencia del anterior en
**una sola cosa**, así una diferencia se puede atribuir a esa cosa y a ninguna otra.

El nombre dice qué le falta al método: un asterisco marca un componente ausente,
dos marcan dos, y un nombre sin marca es el método completo.

| id | nombre | adaptación | ponderación | término local | instancias que usa |
|---|---|---|---|---|---|
| `A` | `Baseline` | — | — | — | todas |
| `C` | `CREDA*` | CREDA | — | — | todas |
| `D` | `CREDA` | CREDA | sí | — | todas |
| `B` | `MIL-Baseline` | — | — | — | todas |
| `E` | `MIL-CREDA**` | MIL-CREDA | — | — | todas |
| `F` | `MIL-CREDA*` | MIL-CREDA | sí | — | todas |
| `G` | `MIL-CREDA` | MIL-CREDA | sí | sí | todas |
| `SU` | `MIL-CREDA-U` | MIL-CREDA | sí | sí | 10, selección regular |
| `SA` | `MIL-CREDA-A` | MIL-CREDA | sí | sí | 10, selección arbitraria |
| `SK` | `MIL-CREDA-K` | MIL-CREDA | sí | sí | 10, las de mayor atención |

Los últimos tres mantienen fijo el presupuesto de instancias y se diferencian solo
en la regla que lo gasta, así que `MIL-CREDA-U → MIL-CREDA-K` y
`MIL-CREDA-A → MIL-CREDA-K` son atribuibles a la regla. `MIL-CREDA-K → MIL-CREDA`
es la pregunta aparte de cuánto cuesta el presupuesto en sí.

Todo lo de abajo está acotado por `config.py`, y solo dos constantes separan esta
corrida de la completa. **Leer el encabezado de cada tabla antes que sus números**:
por debajo del piso de repeticiones declarado no se otorga ningún veredicto y el
motivo queda estampado.

In [1]:
# Bootstrap: locate the repository wherever this is running, and import from it.
# Local, Colab and Kaggle differ only in where the checkout sits.
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

repository: /Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation


In [2]:
import json
import time

from MIL_CREDA_Benchmark import config, harness

device = harness.resolve_device()
reduction = harness.Reduction(device=str(device), environment=harness.environment())

shape = config.sizing()
print(json.dumps(shape, indent=2))
print()
print(harness.header(reduction))
print()
print("environment:", reduction.environment["platform"],
      "| torch", reduction.environment["torch"],
      "| self-hosted" if reduction.environment["selfHosted"] else "| hosted runtime")

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "arms": 10,
  "transfers": 6,
  "seeds": 1,
  "runs": 60,
  "epochs": 3,
  "stepsPerEpoch": 7,
  "imagesPerStep": 300,
  "imagePassesPerRun": 12600,
  "verdictsMeaningful": false
}

setting=trained  backbone=resnet18  bags=100x30  split=64/36  epochs=3  seeds=1  device=mps  revision=research-concept-r16.md
!! 1 repetition(s): the dispersion is zero, so the threshold is zero and every row below declares a winner from a bare difference. These are point estimates, not verdicts.

environment: macOS-26.5.2-arm64-arm-64bit | torch 2.13.0 | self-hosted


In [3]:
# One run, timed, before committing to the whole grid. An estimate of the cost is
# cheaper than the cost, so it happens first.
from MIL_CREDA_Benchmark import bags

material = {role: bags.build(code, config.DATA_CACHE, config.SEEDS[0])
            for role, code in zip(("source", "target"), config.TRANSFERS[0])}
probe = harness.run_one("G", config.TRANSFERS[0], config.SEEDS[0],
                        reduction, device, material)
per_run = probe["seconds"]
full = len(config.ARMS) * len(config.TRANSFERS) * 30 * per_run * 20 / reduction.epochs
print(f"one full arm, {reduction.epochs} epochs: {per_run:.1f}s")
print(f"this grid ({shape['runs']} runs): about {shape['runs'] * per_run / 60:.0f} min")
print(f"at 20 epochs and 30 seeds: about {full / 3600:.0f} h")
del material, probe

one full arm, 3 epochs: 17.3s
this grid (60 runs): about 17 min
at 20 epochs and 30 seeds: about 58 h


## La corrida

Una línea por transferencia, no una por corrida: con treinta semillas la lista
completa serían mil ochocientas líneas y todo lo que dicen ya está en las tablas
de más abajo. Esta salida existe para saber que la campaña sigue viva, nada más.

In [4]:
seen = {"n": 0}

def progress(line: str) -> None:
    seen["n"] += 1
    if seen["n"] % len(config.ARMS) == 0:
        print(f"  {seen['n']:>5}/{shape['runs']} corridas "
              f"({(time.perf_counter() - started) / 60:.1f} min)")

started = time.perf_counter()
summary = harness.campaign(reduction, device, progress=progress)
runs = [json.loads(line) for line in
        (config.RESULTS / "runs.jsonl").read_text().splitlines() if line.strip()]
print(f"\ncampaña terminada en {(time.perf_counter() - started) / 60:.1f} min, "
      f"{len(runs)} corridas")

     10/60 corridas (2.2 min)


     20/60 corridas (4.3 min)


     30/60 corridas (6.4 min)


     40/60 corridas (8.4 min)


     50/60 corridas (10.7 min)


     60/60 corridas (12.3 min)



campaña terminada en 12.3 min, 60 corridas


## 1 · Tiempo de entrenamiento

**Qué mide:** los segundos que tarda una corrida completa de cada método en cada
transferencia, con la misma máquina, el mismo backbone y la misma cantidad de
pasos. **Por qué:** un método que gana pagando diez veces el cómputo no gana lo
mismo que uno que gana gratis, y el costo es lo único que se compara limpio
aunque las dos familias predigan sobre unidades distintas. **Más bajo es mejor.**

In [5]:
from MIL_CREDA_Benchmark import tables

print(tables.render(runs, "seconds", summary["reduction"], extras=("max",)))
print()
print(tables.conclusion(runs, "seconds", summary["reduction"]))

tiempo de entrenamiento (s)  ·  resnet18  ·  3 épocas  ·  1 semilla(s)  ·  research-concept-r16.md
!! 1 repetición(es): el ± de abajo es cero por construcción, no por acuerdo. Son estimaciones puntuales, no resultados.
!! piloto: el protocolo declara 30 repeticiones y 20 épocas. Nada de esto es un resultado.

Método                    M->U            U->M            M->S            S->M            U->S            S->U    Prom.     máx
Baseline           5.86 ± 0.00     5.46 ± 0.00     5.56 ± 0.00     5.58 ± 0.00     5.92 ± 0.00     4.23 ± 0.00     5.43    5.43
CREDA*            12.20 ± 0.00    10.82 ± 0.00    10.13 ± 0.00    10.35 ± 0.00    10.92 ± 0.00     8.74 ± 0.00    10.53   10.53
CREDA             10.59 ± 0.00    10.05 ± 0.00     9.99 ± 0.00    10.60 ± 0.00    10.22 ± 0.00     8.01 ± 0.00     9.91    9.91
MIL-Baseline       6.21 ± 0.00     5.64 ± 0.00     5.75 ± 0.00     5.72 ± 0.00     5.68 ± 0.00     4.30 ± 0.00     5.55    5.55
MIL-CREDA**       13.01 ± 0.00    15.53 ± 0.00   

## 2 · Exactitud en el dominio fuente

**Qué mide:** la proporción de bolsas de evaluación del dominio **fuente** que el
método clasifica bien. **Por qué va primero:** es el complemento, y sin él la
tabla de destino no distingue un éxito de una degeneración — un método que sube en
destino rompiendo la fuente aparece como ganador si solo se mira una tabla.
**Más alto es mejor.**

In [6]:
print(tables.render(runs, "sourceAccuracy", summary["reduction"]))
print()
print(tables.conclusion(runs, "sourceAccuracy", summary["reduction"]))

exactitud en fuente (%)  ·  resnet18  ·  3 épocas  ·  1 semilla(s)  ·  research-concept-r16.md
!! 1 repetición(es): el ± de abajo es cero por construcción, no por acuerdo. Son estimaciones puntuales, no resultados.
!! piloto: el protocolo declara 30 repeticiones y 20 épocas. Nada de esto es un resultado.
la exactitud se mueve de a 2.78 puntos sobre 36 bolsas de evaluación

Método                    M->U            U->M            M->S            S->M            U->S            S->U    Prom.     máx    peso
Baseline           100.0 ± 0.0     100.0 ± 0.0     100.0 ± 0.0      41.7 ± 0.0     100.0 ± 0.0      50.0 ± 0.0     81.9    81.9   0.000
CREDA*             100.0 ± 0.0     100.0 ± 0.0     100.0 ± 0.0      41.7 ± 0.0     100.0 ± 0.0      50.0 ± 0.0     81.9    81.9   0.030
CREDA              100.0 ± 0.0     100.0 ± 0.0     100.0 ± 0.0      38.9 ± 0.0     100.0 ± 0.0      50.0 ± 0.0     81.5    81.5   0.273
MIL-Baseline        94.4 ± 0.0     100.0 ± 0.0     100.0 ± 0.0      27.8 ± 0.0  

### 2b · Los peldaños en fuente

**Qué mide:** la diferencia entre los dos métodos de cada peldaño, transferencia
por transferencia. El signo va hacia el método de la derecha, así que un valor
positivo quiere decir que el de la derecha quedó por encima. **Por qué:** la tabla
anterior dice quién está adelante; solo esta dice **qué componente** lo puso ahí,
porque los dos métodos de un peldaño se diferencian en una sola cosa.
**Más alto es mejor** (a favor del método de la derecha).

In [7]:
print(tables.render_rungs(summary, "sourceAccuracy"))
print()
print(tables.conclusion_rungs(summary, "sourceAccuracy"))

peldaños · exactitud en fuente (%) · diferencia con signo hacia la derecha
!! 1 repetición(es): el ± de abajo es cero por construcción, no por acuerdo. Son estimaciones puntuales, no resultados.
!! piloto: el protocolo declara 30 repeticiones y 20 épocas. Nada de esto es un resultado.

Peldaño                           M->U      U->M      M->S      S->M      U->S      S->U     Prom.   a favor
Baseline → MIL-Baseline           -5.6      +0.0      +0.0     -13.9      -5.6     -13.9      -6.50/6       
                                what the bag representation buys, with adaptation off
Baseline → CREDA*                 +0.0      +0.0      +0.0      +0.0      +0.0      +0.0      +0.00/6       
                                what CREDA's alignment buys, unweighted
CREDA* → CREDA                    +0.0      +0.0      +0.0      -2.8      +0.0      +0.0      -0.50/6       
                                what confidence weighting buys in CREDA
MIL-Baseline → MIL-CREDA**        +2.8      -2.

## 3 · Exactitud en el dominio destino

**Qué mide:** la proporción de bolsas de evaluación del dominio **destino** —
aquel cuyas etiquetas el método nunca vio — que clasifica bien. **Por qué:** es la
pregunta del problema. **Más alto es mejor**, y se lee junto a la tabla de fuente,
nunca sola.

In [8]:
print(tables.render(runs, "targetAccuracy", summary["reduction"]))
print()
print(tables.conclusion(runs, "targetAccuracy", summary["reduction"]))

exactitud en destino (%)  ·  resnet18  ·  3 épocas  ·  1 semilla(s)  ·  research-concept-r16.md
!! 1 repetición(es): el ± de abajo es cero por construcción, no por acuerdo. Son estimaciones puntuales, no resultados.
!! piloto: el protocolo declara 30 repeticiones y 20 épocas. Nada de esto es un resultado.
la exactitud se mueve de a 2.78 puntos sobre 36 bolsas de evaluación

Método                    M->U            U->M            M->S            S->M            U->S            S->U    Prom.     máx    peso
Baseline            77.8 ± 0.0      97.2 ± 0.0       8.3 ± 0.0      72.2 ± 0.0      19.4 ± 0.0      47.2 ± 0.0     53.7    53.7   0.000
CREDA*              83.3 ± 0.0     100.0 ± 0.0       8.3 ± 0.0      72.2 ± 0.0      22.2 ± 0.0      47.2 ± 0.0     55.6    55.6   0.030
CREDA               72.2 ± 0.0      97.2 ± 0.0      13.9 ± 0.0      52.8 ± 0.0      19.4 ± 0.0      36.1 ± 0.0     48.6    48.6   0.273
MIL-Baseline        47.2 ± 0.0      83.3 ± 0.0      22.2 ± 0.0      38.9 ± 0.0 

### 3b · Los peldaños en destino

**Qué mide:** lo mismo que 2b, sobre el dominio destino. **Por qué:** acá es donde
se lee qué aporta cada componente al problema que se quiere resolver.
**Más alto es mejor** (a favor del método de la derecha).

In [9]:
print(tables.render_rungs(summary, "targetAccuracy"))
print()
print(tables.conclusion_rungs(summary, "targetAccuracy"))

peldaños · exactitud en destino (%) · diferencia con signo hacia la derecha
!! 1 repetición(es): el ± de abajo es cero por construcción, no por acuerdo. Son estimaciones puntuales, no resultados.
!! piloto: el protocolo declara 30 repeticiones y 20 épocas. Nada de esto es un resultado.

Peldaño                           M->U      U->M      M->S      S->M      U->S      S->U     Prom.   a favor
Baseline → MIL-Baseline          -30.6     -13.9     +13.9     -33.3      -8.3     -13.9     -14.41/6       
                                what the bag representation buys, with adaptation off
Baseline → CREDA*                 +5.6      +2.8      +0.0      +0.0      +2.8      +0.0      +1.93/6       
                                what CREDA's alignment buys, unweighted
CREDA* → CREDA                   -11.1      -2.8      +5.6     -19.4      -2.8     -11.1      -6.91/6       
                                what confidence weighting buys in CREDA
MIL-Baseline → MIL-CREDA**       +11.1      +2

## 4 · La lectura transversal

**Qué mide:** cada peldaño promediado sobre las seis transferencias a la vez, con
su dispersión y cuántas de las seis se inclinan hacia la derecha. **Por qué:**
ninguna transferencia de este tamaño resuelve por sí sola una diferencia chica —
36 bolsas de evaluación mueven la exactitud de a 2,78 puntos — así que lo que
carga peso no es el valor de una, sino que las seis coincidan. Un peldaño que se
inclina igual en 6 de 6 dice algo; uno que queda 3 y 3 no dice nada.
**Más alto es mejor** (a favor del método de la derecha).

In [10]:
print(harness.render_panorama(summary))

panorama over 6 transfers, 1 seed(s) each

rung    metric              difference     leans  reading
A->B    targetAccuracy    -0.1435 ± 0.070     1/6    what the bag representation buys, with adaptation off
A->B    sourceAccuracy    -0.0648 ± 0.026     0/6    what the bag representation buys, with adaptation off
A->C    targetAccuracy    +0.0185 ± 0.009     3/6    what CREDA's alignment buys, unweighted
A->C    sourceAccuracy    +0.0000 ± 0.000     0/6    what CREDA's alignment buys, unweighted
C->D    targetAccuracy    -0.0694 ± 0.036     1/6    what confidence weighting buys in CREDA
C->D    sourceAccuracy    -0.0046 ± 0.005     0/6    what confidence weighting buys in CREDA
B->E    targetAccuracy    +0.0417 ± 0.016     5/6    what the global term buys, unweighted
B->E    sourceAccuracy    -0.0185 ± 0.015     1/6    what the global term buys, unweighted
E->F    targetAccuracy    -0.0417 ± 0.017     0/6    what confidence weighting buys in MIL-CREDA
E->F    sourceAccuracy    +0.0139 

## 5 · Las curvas

Mediana entre semillas con banda intercuartil, nunca una corrida sola: una
trayectoria única no puede mostrar si la forma es del método o del sorteo.

In [11]:
from MIL_CREDA_Benchmark import figures

for name, draw in (("supervised", figures.supervised_curves),
                   ("adaptation", figures.adaptation_curves),
                   ("contribution", figures.contribution_curves)):
    draw(config.RESULTS / "curves" / f"{name}.png")
    print(f"escrita: curves/{name}.png")

escrita: curves/supervised.png


escrita: curves/adaptation.png


escrita: curves/contribution.png


## 6 · El registro

Generado junto con los resultados y nunca escrito a mano. Un resumen escrito a
mano es una segunda fuente de verdad: se desactualiza en silencio y se le cree
igual.

In [12]:
bloques = [
    harness.header(reduction),
    tables.render(runs, "seconds", summary["reduction"], extras=("max",)),
    tables.conclusion(runs, "seconds", summary["reduction"]),
    tables.render(runs, "sourceAccuracy", summary["reduction"]),
    tables.conclusion(runs, "sourceAccuracy", summary["reduction"]),
    tables.render_rungs(summary, "sourceAccuracy"),
    tables.conclusion_rungs(summary, "sourceAccuracy"),
    tables.render(runs, "targetAccuracy", summary["reduction"]),
    tables.conclusion(runs, "targetAccuracy", summary["reduction"]),
    tables.render_rungs(summary, "targetAccuracy"),
    tables.conclusion_rungs(summary, "targetAccuracy"),
    harness.render_panorama(summary),
]
(config.RESULTS / "report.txt").write_text("\n\n".join(bloques), encoding="utf-8")

(config.RESULTS / "report.md").write_text("\n\n".join([
    f"# Fase uno — {config.REVISION}",
    "## 1 · Tiempo de entrenamiento (más bajo es mejor)",
    tables.render(runs, "seconds", summary["reduction"], markdown=True, extras=("max",)),
    tables.conclusion(runs, "seconds", summary["reduction"]),
    "## 2 · Exactitud en fuente (más alto es mejor)",
    tables.render(runs, "sourceAccuracy", summary["reduction"], markdown=True),
    tables.conclusion(runs, "sourceAccuracy", summary["reduction"]),
    "### 2b · Peldaños en fuente",
    tables.render_rungs(summary, "sourceAccuracy", markdown=True),
    tables.conclusion_rungs(summary, "sourceAccuracy"),
    "## 3 · Exactitud en destino (más alto es mejor)",
    tables.render(runs, "targetAccuracy", summary["reduction"], markdown=True),
    tables.conclusion(runs, "targetAccuracy", summary["reduction"]),
    "### 3b · Peldaños en destino",
    tables.render_rungs(summary, "targetAccuracy", markdown=True),
    tables.conclusion_rungs(summary, "targetAccuracy"),
]), encoding="utf-8")

print("escritos:")
for path in sorted(config.RESULTS.iterdir()):
    print(" ", path.relative_to(config.REPOSITORY))

escritos:
  MIL-CREDA/Results/Benchmark/curves
  MIL-CREDA/Results/Benchmark/latent
  MIL-CREDA/Results/Benchmark/latent.json
  MIL-CREDA/Results/Benchmark/latent.md
  MIL-CREDA/Results/Benchmark/report.md
  MIL-CREDA/Results/Benchmark/report.txt
  MIL-CREDA/Results/Benchmark/runs.jsonl
  MIL-CREDA/Results/Benchmark/summary.json


In [13]:
# El sello: contra qué código corrió este informe. Sin él, un informe viejo y uno
# recién generado se ven idénticos y el viejo se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())

SOURCES-SHA256 0cf518226878861386cf8b7edb096149e50a100ef0391d65b1599463edcae5bb
